# App rápida (sin reentrenar)
Descarga los resultados ya calculados desde el **Release** del repositorio (`resultados_para_colab.zip`, con el contenido de `data/processed`) y levanta la app.

*Entorno de ejecución → Ejecutar todas* y abre la liga `https://….trycloudflare.com` que se imprime al final.

In [ ]:
!git clone -q https://github.com/AndreaHortaHernandez/ParticipacionCiudadanaINE.git
%cd ParticipacionCiudadanaINE
!pip -q install -r requirements.txt

### Resultados precalculados
Se descargan del último Release. Para usar una versión específica, cambia `latest/download` por `download/v1.0` (o el tag que corresponda).

In [ ]:
import os, glob, shutil, zipfile, urllib.request

URL = 'https://github.com/AndreaHortaHernandez/ParticipacionCiudadanaINE/releases/latest/download/resultados_para_colab.zip'
urllib.request.urlretrieve(URL, 'resultados_para_colab.zip')
zipfile.ZipFile('resultados_para_colab.zip').extractall('tmp_zip')

# Acepta el zip tanto con los archivos en la raíz como dentro de data/processed/
encontrados = glob.glob('tmp_zip/**/riesgo_*.parquet', recursive=True)
assert encontrados, 'El zip no trae ningún riesgo_*.parquet'
os.makedirs('data/processed', exist_ok=True)
shutil.copytree(os.path.dirname(encontrados[0]), 'data/processed', dirs_exist_ok=True)
shutil.rmtree('tmp_zip'); os.remove('resultados_para_colab.zip')
print(sorted(os.listdir('data/processed')))

### Cartografía y agente
Solo se generan si no vienen en el zip. Si la cartografía falla, la app funciona igual, pero sin la pestaña de mapa.

In [ ]:
%%bash
if [ -f data/processed/geo/entidades.geojson ]; then
  echo 'Cartografía incluida en el zip: se omite.'
else
  mkdir -p data/geo && cd data/geo
  for f in SECCION MUNICIPIO ENTIDAD; do curl -sL -A 'Mozilla/5.0' -O "https://pautas.ine.mx/transparencia/mapas/cob_2025/$f.zip" && mkdir -p $f && unzip -oq $f.zip -d $f; rm -f $f.zip; done
  cd ../..
  python -W ignore -m src.geo || echo '⚠️ No se pudo generar la cartografía: la app correrá sin mapas.'
fi
if [ -f data/processed/intervenciones_2027.parquet ]; then
  echo 'Plan del agente incluido en el zip: se omite.'
else
  python -m src.agent --anio 2027 --presupuesto-pct 10
fi

### App
Abre la liga `https://….trycloudflare.com` que se imprime. Funciona mientras esta sesión siga abierta.

In [ ]:
%%bash
wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared
nohup streamlit run app.py --server.port 8501 --server.headless true > streamlit.log 2>&1 &
nohup ./cloudflared tunnel --url http://localhost:8501 > cloudflared.log 2>&1 &
# Espera hasta 60 s a que aparezca la liga
for i in $(seq 1 30); do
  URL=$(grep -oE 'https://[a-z0-9-]+\.trycloudflare\.com' cloudflared.log | head -1)
  [ -n "$URL" ] && break; sleep 2
done
echo "${URL:-No apareció la liga; revisa cloudflared.log}"